# SongGeneration (LeVo): Text-to-Music on AWS Trainium2

This notebook demonstrates how to run [SongGeneration](https://github.com/tencent-ailab/songgeneration) (Tencent AI Lab's LeVo) on AWS Trainium2 using Neuron SDK. SongGeneration is a multi-stage text-to-music pipeline that generates stereo 48kHz vocal music from lyrics and text descriptions.

## Architecture

The pipeline has three compiled stages:

| Stage | Component | Params | Description |
|-------|-----------|--------|-------------|
| 1. LeLM | Dual-Llama AR (28L + 12L) | 2.83B | Generates music codec tokens from lyrics |
| 2. Diffusion | GPT2-RoPE CFM (16L) | 1.1B | Converts tokens to latents (10 Euler steps) |
| 3. VAE | Stable Audio decoder | 169M | Converts latents to stereo 48kHz audio |

## Neuron Optimizations

- **On-device KV cache**: LeLM uses `torch.scatter` for in-HBM KV updates (no PCIe round-trips)
- **Prefill NEFF**: Processes 512 condition tokens in one Neuron call (45% LeLM speedup)
- **NxDI TP=2 mode**: Optional tensor-parallel decode for 2x LeLM speedup

## Requirements

- **Instance**: trn2.3xlarge (LNC=2, 4 NeuronCores)
- **AMI**: Deep Learning AMI Neuron (Ubuntu 24.04) 20260227 (SDK 2.28)
- **Storage**: NVMe mounted at `/mnt/models`
- **Venv**: `/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/`

## What This Notebook Covers

1. **Setup**: Install dependencies, download models, apply patches
2. **Compile**: Build the Neuron pipeline (~20 min first time)
3. **Generate**: Produce 15s of English vocal music from lyrics
4. **Benchmark**: Multi-seed performance measurement

For component accuracy validation (GPT2, VAE cosine similarity vs CPU), see `test/integration/test_model.py`.

## 1. Setup

Install dependencies, download model weights, and apply required patches. Run once per instance.

In [1]:
# Configuration
import os

BASE_DIR = "/mnt/models/songgeneration"
CKPT_DIR = "/mnt/models/ckpt"

In [2]:
%%bash -s "$BASE_DIR" "$CKPT_DIR"
BASE_DIR=$1
CKPT_DIR=$2

echo "=== Installing Python dependencies ==="
pip install -q accelerate flashy alias-free-torch descript-audio-codec \
    k-diffusion vector-quantize-pytorch einops-exts x-transformers \
    diffusers==0.37.0 peft==0.18.0 lightning openunmix huggingface_hub
pip install -q protobuf==5.29.3  # Must be after descript-audio-codec

echo "=== Setting up directories ==="
sudo mkdir -p "${BASE_DIR}" "${CKPT_DIR}"
sudo chown -R ubuntu:ubuntu "${BASE_DIR}" "${CKPT_DIR}"
echo "Done"

=== Installing Python dependencies ===


ERROR: pip's dependency resolver does not currently take into account all the packages that are

 installed. This behaviour is the source of the following dependency conflicts.
ray 2.54.0 requires 

protobuf>=3.20.3, but you have protobuf 3.19.6 which is incompatible.
neuronx-cc 2.23.6484.0+3b61258

3 requires networkx~=2.6, but you have networkx 3.6.1 which is incompatible.
neuronx-cc 2.23.6484.0+

...

23.6484.0+3b612583 requires numpy>=2.0.0; python_version >= "3.12", but you have numpy 1.26.4 which 

is incompatible.
descript-audiotools 0.7.2 requires protobuf<3.20,>=3.9.2, but you have protobuf 5.2

9.3 which is incompatible.


=== Setting up directories ===


Done


In [3]:
%%bash -s "$BASE_DIR" "$CKPT_DIR"
BASE_DIR=$1
CKPT_DIR=$2

echo "=== Downloading SongGeneration-base-new (English + Chinese) ==="
if [ ! -f "${CKPT_DIR}/songgeneration_base_new/model.pt" ]; then
    python -c "
from huggingface_hub import snapshot_download
snapshot_download('lglg666/SongGeneration-base-new',
                  local_dir='${CKPT_DIR}/songgeneration_base_new',
                  ignore_patterns=['*.md'])
"
else
    echo "  Already exists, skipping."
fi

echo "=== Downloading shared assets (diffusion, VAE, tokenizer) ==="
if [ ! -f "${CKPT_DIR}/songgeneration_base/ckpt/model_septoken/model_2.safetensors" ]; then
    python -c "
from huggingface_hub import snapshot_download
snapshot_download('tencent/SongGeneration',
                  local_dir='${CKPT_DIR}/songgeneration_base',
                  ignore_patterns=['*.md'])
"
else
    echo "  Already exists, skipping."
fi

echo "=== Cloning codeclm source ==="
if [ ! -d "${BASE_DIR}/codeclm_repo" ]; then
    git clone https://github.com/tencent-ailab/songgeneration.git "${BASE_DIR}/codeclm_repo"
else
    echo "  Already exists, skipping."
fi

echo "=== Pulling language-aware prompt audio (git-lfs) ==="
cd "${BASE_DIR}/codeclm_repo"
git lfs pull --include='tools/new_auto_prompt.pt'
ls -lh tools/new_auto_prompt.pt

echo "=== Setting up symlinks and dirs ==="
cd "${BASE_DIR}"
cp -r codeclm_repo/codeclm . 2>/dev/null || true
rm -f ckpt third_party
ln -sf "${CKPT_DIR}/songgeneration_base/third_party" third_party
ln -sf "${CKPT_DIR}/songgeneration_base/ckpt" ckpt
mkdir -p conf && cp codeclm_repo/conf/vocab.yaml conf/vocab.yaml 2>/dev/null || true

echo "=== Verifying files ==="
ls -lh "${CKPT_DIR}/songgeneration_base_new/model.pt"
ls -lh "${BASE_DIR}/ckpt/model_septoken/model_2.safetensors"
ls -lh "${BASE_DIR}/codeclm_repo/tools/new_auto_prompt.pt"
echo "=== Done ==="

=== Downloading SongGeneration-base-new (English + Chinese) ===


  Already exists, skipping.
=== Downloading shared assets (diffusion, VAE, tokenizer) ===
  Already 

exists, skipping.
=== Cloning codeclm source ===
  Already exists, skipping.
=== Pulling language-aw

are prompt audio (git-lfs) ===


-rw-rw-r-- 1 ubuntu ubuntu 15M Mar 25 14:05 tools/new_auto_prompt.pt


...

-rw-rw-r-- 1 ubuntu ubuntu 3.6G Mar 25 14:03 /mnt/models/songgeneration/ckpt/model_septoken/model_2.

safetensors


-rw-rw-r-- 1 ubuntu ubuntu 15M Mar 25 14:05 /mnt/models/songgeneration/codeclm_repo/tools/new_auto_p

rompt.pt


=== Done ===


In [4]:
%%bash -s "$BASE_DIR"
BASE_DIR=$1

echo "=== Applying required patches ==="

# 1. SequenceSummary stub (transformers modeling_utils)
UTILS_FILE=$(python3 -c "import transformers.modeling_utils; print(transformers.modeling_utils.__file__)")
if ! grep -q 'class SequenceSummary' "$UTILS_FILE"; then
    echo '
class SequenceSummary:
    pass' >> "$UTILS_FILE"
    echo "  Patched SequenceSummary"
else
    echo "  SequenceSummary already patched"
fi

# 2. Flash attention import fix
find "${BASE_DIR}/codeclm/" -name "*.py" -exec sed -i "s/is_flash_attn_available/is_flash_attn_2_available/g" {} +
echo "  Patched flash attention imports"

# 3. Remove transformers version assertion
sed -i "/assert.*transformers.*version/d" "${BASE_DIR}/codeclm/models/levo.py" 2>/dev/null || true
echo "  Removed transformers version assertion"

# 4. Clear stale __pycache__
find "${BASE_DIR}" -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null || true
echo "  Cleared __pycache__"

echo "=== Patches applied ==="

=== Applying required patches ===


/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/transformers/utils/hu

b.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Trans

formers. Use `HF_HOME` instead.
  warnings.warn(


  SequenceSummary already patched


  Patched flash attention imports


  Removed transformers version assertion


  Cleared __pycache__
=== Patches applied ===


In [5]:
# Set up Python paths
import sys

BASE_DIR = "/mnt/models/songgeneration"
CKPT_DIR = "/mnt/models/ckpt"

paths_to_add = [
    f"{BASE_DIR}/codeclm/tokenizer/",
    BASE_DIR,
    f"{BASE_DIR}/codeclm/tokenizer/Flow1dVAE/",
    f"{BASE_DIR}/contrib_songgen/src/",
]
for p in paths_to_add:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["TRANSFORMERS_CACHE"] = f"{BASE_DIR}/third_party/hub"
os.environ["NEURON_COMPILE_CACHE_URL"] = "/mnt/models/neuron_cache"

print("Python paths configured")

Python paths configured


In [6]:
# Verify Neuron SDK
import torch
import torch_neuronx

print(f"PyTorch: {torch.__version__}")
print(f"torch-neuronx: {torch_neuronx.__version__}")

!neuron-ls

PyTorch: 2.9.0+cu128
torch-neuronx: 2.9.0.2.12.22436+0f1dac25
instance-type: trn2.3xlarge
instance-id: i-0b9c0855e8f3c7d57
logical-neuroncore-config: 2
+--------+--------+----------+--------+--------------+----------+------+
| NEURON | NEURON |  NEURON  | NEURON |     PCI      |   CPU    | NUMA |
| DEVICE | CORES  | CORE IDS | MEMORY |     BDF      | AFFINITY | NODE |
+--------+--------+----------+--------+--------------+----------+------+
| 0      | 4      | 0-3      | 96 GB  | 0000:33:00.0 | 0-11     | 0    |
+--------+--------+----------+--------+--------------+----------+------+


## 2. Compile and Generate (Baseline TP=1)

Build the SongGeneration Neuron pipeline with on-device KV cache and prefill optimization. First compilation takes ~20 minutes.

In [7]:
from modeling_songgeneration import SongGenerationNeuron, SongGenerationConfig

config = SongGenerationConfig(
    model_path=f"{CKPT_DIR}/songgeneration_base_new/model.pt",
    config_path=f"{CKPT_DIR}/songgeneration_base_new/config.yaml",
    safetensors_path=f"{BASE_DIR}/ckpt/model_septoken/model_2.safetensors",
    prompt_path=f"{BASE_DIR}/codeclm_repo/tools/new_auto_prompt.pt",
    codeclm_path=BASE_DIR,
    default_duration_sec=15.0,
)

print(f"Model: {config.model_path}")
print(f"Duration: {config.default_duration_sec}s")
print(f"Max seq len: {config.max_seq_len}")
print(f"Prefill len: {config.prefill_len}")

Model: /mnt/models/ckpt/songgeneration_base_new/model.pt
Duration: 15.0s
Max seq len: 512
Prefill len: 512


In [8]:
import time

pipeline = SongGenerationNeuron(config)

print("Compiling pipeline (LeLM + GPT2 + VAE)...")
t0 = time.time()
pipeline.compile()
compile_time = time.time() - t0
print(f"Compilation complete in {compile_time:.0f}s")

Compiling pipeline (LeLM + GPT2 + VAE)...


/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx

/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/moe_fused_tkg.py:49: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/lib/python3.12/site-packages/neuronx_distributed/modules/moe/moe_fused_tkg.py:49: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)


[1/5] Loading LeLM model on CPU...


...

.

.


Compiler status PASS


[+] Loading CPU diffusion components...
Compilation complete.


Compilation complete in 1681s


In [9]:
# Load English prompt audio
auto_prompt = torch.load(
    f"{BASE_DIR}/codeclm_repo/tools/new_auto_prompt.pt",
    map_location="cpu",
    weights_only=False,
)
pipeline._prompt_data = {
    g: auto_prompt[g]["en"] for g in auto_prompt if "en" in auto_prompt[g]
}
print(f"Loaded English prompts for genres: {list(pipeline._prompt_data.keys())}")

# Warmup
pipeline.warmup()
print("Warmup complete")

Loaded English prompts for genres: ['Pop', 'Latin', 'Rock', 'Electronic', 'Metal', 'Country', 'R&B/Soul', 'Ballad', 'Jazz', 'World', 'Hip-Hop', 'Funk', 'Classical', 'Soundtrack', 'Auto']


Warmup complete


In [10]:
# Generate 15s of English vocal music
lyrics = (
    "[intro-short] ; "
    "[verse] Sunlight breaks through morning haze."
    "Golden fields stretch far away."
    "Rivers flow with gentle grace."
    "Finding peace in nature's embrace ; "
    "[chorus] Sing along.Let the music carry you home."
    "Sing along.You were never meant to walk alone ; "
    "[outro-short]"
)

print("Generating 15s audio (seed=42)...")
result = pipeline.generate_timed(lyrics, genre="Pop", duration_sec=15.0, seed=42)

audio = result["audio"]
sr = result["sample_rate"]
t = result["timings"]

print(f"\n--- Baseline TP=1 Results ---")
print(f"Audio shape: {audio.shape} (batch, channels, samples)")
print(f"Sample rate: {sr} Hz")
print(f"LeLM time:   {t['lelm_s']:.1f}s ({t['lelm_steps']} steps, {t['lelm_s']*1000/t['lelm_steps']:.1f} ms/step)")
print(f"Diffusion:   {t['diffusion_s']:.3f}s")
print(f"VAE decode:  {t['vae_s']:.3f}s")
print(f"Total:       {t['total_s']:.1f}s")
print(f"RTF:         {t['total_s']/15.0:.2f}x (< 1.0 = faster than real-time)")

Generating 15s audio (seed=42)...
conditions [ConditioningAttributes(text={'description': "[intro-short] ; [verse] Sunlight breaks through morning haze.Golden fields stretch far away.Rivers flow with gentle grace.Finding peace in nature's embrace ; [chorus] Sing along.Let the music carry you home.Sing along.You were never meant to walk alone ; [outro-short]", 'type_info': ''}, audio={'prompt_audio': AudioCondition(wav=tensor([[[16384,  7293,  1321, 14819, 13185,  4081, 10629,  6545,  1814,  4081,
           5959,  5959,  5959, 13394,  2970,  4975,  4938, 13783, 10997,  5529,
           3957, 12242, 12518,  4278,  2155,  2155,  2155, 11301, 10167, 14452,
          16151,  5437,  2153,  9453, 10597, 10597,  6912, 15968,  8821,  4523,
           1439,  1439,  1439, 13723,  1468,  7186,  5567,  7885,  7186,  1952,
            398,  5777,  6555,  6608, 10680, 12809,   915,  7835,  4712,   888,
          15633,  9235, 12004, 11640,  7649, 13289, 13139, 15746,   809,   809,
          13288,  


--- Baseline TP=1 Results ---
Audio shape: torch.Size([1, 2, 720000]) (batch, channels, samples)
Sample rate: 48000 Hz
LeLM time:   39.8s (1227 steps, 32.4 ms/step)
Diffusion:   0.377s
VAE decode:  0.210s
Total:       40.4s
RTF:         2.69x (< 1.0 = faster than real-time)


In [11]:
# Save output as WAV
import numpy as np
import scipy.io.wavfile

output_path = "/mnt/models/songgeneration_baseline_output.wav"

audio_np = audio.squeeze(0).float().cpu().numpy().T  # (samples, channels)
audio_np = np.clip(audio_np, -1.0, 1.0)
audio_int16 = (audio_np * 32767).astype(np.int16)
scipy.io.wavfile.write(output_path, sr, audio_int16)

# Audio statistics
mono = audio_np.mean(axis=1) if audio_np.ndim > 1 else audio_np
peak = max(abs(mono.max()), abs(mono.min()), 1e-10)
rms = np.sqrt(np.mean((mono / peak * 32767).astype(np.int16).astype(float) ** 2))

print(f"Saved: {output_path}")
print(f"Audio peak: {peak:.4f}")
print(f"Audio RMS: {rms:.0f} (healthy > 1000)")
print(f"Audio std: {audio.std().item():.6f}")

Saved: /mnt/models/songgeneration_baseline_output.wav
Audio peak: 1.0000
Audio RMS: 9422 (healthy > 1000)
Audio std: 0.302435


## 3. Multi-Seed Benchmark

Generate with multiple seeds to measure performance consistency.

In [12]:
seeds = [42, 123, 7]
results_baseline = []

for seed in seeds:
    print(f"\nGenerating with seed={seed}...")
    r = pipeline.generate_timed(lyrics, genre="Pop", duration_sec=15.0, seed=seed)
    t = r["timings"]
    a = r["audio"]
    ms_per_step = t["lelm_s"] * 1000 / t["lelm_steps"]
    
    # Save WAV
    wav_path = f"/mnt/models/songgen_baseline_seed{seed}.wav"
    audio_out = a.squeeze(0).float().cpu().numpy().T
    audio_out = np.clip(audio_out, -1.0, 1.0)
    scipy.io.wavfile.write(wav_path, sr, (audio_out * 32767).astype(np.int16))
    
    results_baseline.append({
        "seed": seed,
        "lelm_s": t["lelm_s"],
        "total_s": t["total_s"],
        "ms_per_step": ms_per_step,
        "steps": t["lelm_steps"],
        "audio_std": a.std().item(),
    })
    print(f"  LeLM: {t['lelm_s']:.1f}s | Total: {t['total_s']:.1f}s | {ms_per_step:.1f} ms/step | RTF: {t['total_s']/15.0:.2f}x")

print("\n--- Baseline TP=1 Summary ---")
print(f"{'Seed':<8} {'LeLM':>8} {'Total':>8} {'ms/step':>10} {'RTF':>8}")
for r in results_baseline:
    print(f"{r['seed']:<8} {r['lelm_s']:>7.1f}s {r['total_s']:>7.1f}s {r['ms_per_step']:>9.1f} {r['total_s']/15.0:>7.2f}x")


Generating with seed=42...
conditions [ConditioningAttributes(text={'description': "[intro-short] ; [verse] Sunlight breaks through morning haze.Golden fields stretch far away.Rivers flow with gentle grace.Finding peace in nature's embrace ; [chorus] Sing along.Let the music carry you home.Sing along.You were never meant to walk alone ; [outro-short]", 'type_info': ''}, audio={'prompt_audio': AudioCondition(wav=tensor([[[16384,  7293,  1321, 14819, 13185,  4081, 10629,  6545,  1814,  4081,
           5959,  5959,  5959, 13394,  2970,  4975,  4938, 13783, 10997,  5529,
           3957, 12242, 12518,  4278,  2155,  2155,  2155, 11301, 10167, 14452,
          16151,  5437,  2153,  9453, 10597, 10597,  6912, 15968,  8821,  4523,
           1439,  1439,  1439, 13723,  1468,  7186,  5567,  7885,  7186,  1952,
            398,  5777,  6555,  6608, 10680, 12809,   915,  7835,  4712,   888,
          15633,  9235, 12004, 11640,  7649, 13289, 13139, 15746,   809,   809,
          13288,   508, 

  LeLM: 39.7s | Total: 40.3s | 32.4 ms/step | RTF: 2.69x

Generating with seed=123...
conditions [ConditioningAttributes(text={'description': "[intro-short] ; [verse] Sunlight breaks through morning haze.Golden fields stretch far away.Rivers flow with gentle grace.Finding peace in nature's embrace ; [chorus] Sing along.Let the music carry you home.Sing along.You were never meant to walk alone ; [outro-short]", 'type_info': ''}, audio={'prompt_audio': AudioCondition(wav=tensor([[[16384,  7293,  1321, 14819, 13185,  4081, 10629,  6545,  1814,  4081,
           5959,  5959,  5959, 13394,  2970,  4975,  4938, 13783, 10997,  5529,
           3957, 12242, 12518,  4278,  2155,  2155,  2155, 11301, 10167, 14452,
          16151,  5437,  2153,  9453, 10597, 10597,  6912, 15968,  8821,  4523,
           1439,  1439,  1439, 13723,  1468,  7186,  5567,  7885,  7186,  1952,
            398,  5777,  6555,  6608, 10680, 12809,   915,  7835,  4712,   888,
          15633,  9235, 12004, 11640,  7649, 1

  LeLM: 39.7s | Total: 40.3s | 32.3 ms/step | RTF: 2.68x

Generating with seed=7...
conditions [ConditioningAttributes(text={'description': "[intro-short] ; [verse] Sunlight breaks through morning haze.Golden fields stretch far away.Rivers flow with gentle grace.Finding peace in nature's embrace ; [chorus] Sing along.Let the music carry you home.Sing along.You were never meant to walk alone ; [outro-short]", 'type_info': ''}, audio={'prompt_audio': AudioCondition(wav=tensor([[[16384,  7293,  1321, 14819, 13185,  4081, 10629,  6545,  1814,  4081,
           5959,  5959,  5959, 13394,  2970,  4975,  4938, 13783, 10997,  5529,
           3957, 12242, 12518,  4278,  2155,  2155,  2155, 11301, 10167, 14452,
          16151,  5437,  2153,  9453, 10597, 10597,  6912, 15968,  8821,  4523,
           1439,  1439,  1439, 13723,  1468,  7186,  5567,  7885,  7186,  1952,
            398,  5777,  6555,  6608, 10680, 12809,   915,  7835,  4712,   888,
          15633,  9235, 12004, 11640,  7649, 132

  LeLM: 39.5s | Total: 40.0s | 32.2 ms/step | RTF: 2.67x

--- Baseline TP=1 Summary ---
Seed         LeLM    Total    ms/step      RTF
42          39.7s    40.3s      32.4    2.69x
123         39.7s    40.3s      32.3    2.68x
7           39.5s    40.0s      32.2    2.67x


## 4. Results Summary

In [13]:
import statistics

avg_lelm = statistics.mean([r["lelm_s"] for r in results_baseline])
avg_total = statistics.mean([r["total_s"] for r in results_baseline])
avg_ms = statistics.mean([r["ms_per_step"] for r in results_baseline])

print("=" * 60)
print("SongGeneration Neuron Benchmark Results")
print("=" * 60)
print(f"Instance:    trn2.3xlarge (LNC=2)")
print(f"SDK:         Neuron SDK 2.28")
print(f"Model:       SongGeneration-base-new (English + Chinese)")
print(f"Duration:    15s stereo 48kHz")
print(f"")
print(f"--- Baseline TP=1 Performance (avg of {len(seeds)} seeds) ---")
print(f"LeLM time:       {avg_lelm:.1f}s")
print(f"Total time:      {avg_total:.1f}s")
print(f"Per-step:        {avg_ms:.1f} ms")
print(f"RTF:             {avg_total/15.0:.2f}x")
print(f"")
print(f"Compilation:     {compile_time:.0f}s")
print(f"")
print(f"Component accuracy validated by pytest test suite:")
print(f"  GPT2 cosine sim vs CPU: > 0.98 (PASS)")
print(f"  VAE cosine sim vs CPU:  > 0.98 (PASS)")
print(f"  VAE SNR vs CPU:         > 20 dB (PASS)")
print("=" * 60)

SongGeneration Neuron Benchmark Results
Instance:    trn2.3xlarge (LNC=2)
SDK:         Neuron SDK 2.28
Model:       SongGeneration-base-new (English + Chinese)
Duration:    15s stereo 48kHz

--- Baseline TP=1 Performance (avg of 3 seeds) ---
LeLM time:       39.6s
Total time:      40.2s
Per-step:        32.3 ms
RTF:             2.68x

Compilation:     1681s

Component accuracy validated by pytest test suite:
  GPT2 cosine sim vs CPU: > 0.98 (PASS)
  VAE cosine sim vs CPU:  > 0.98 (PASS)
  VAE SNR vs CPU:         > 20 dB (PASS)
